# 07 — Conformal Prediction avec MAPIE ★★
**Projet LendingClub | Membre 2**

| Étape | Contenu |
|---|---|
| 1 | Chargement données et modèle calibré |
| 2 | Théorie de la Conformal Prediction |
| 3 | MAPIE — intervalles garantis à 90% et 95% |
| 4 | Analyse des clients en « zone grise » |
| 5 | Stratégie de décision avec incertitude |
| 6 | Comparaison largeur des intervalles |

---

## Pourquoi la Conformal Prediction ?

Un modèle bien calibré donne de **bonnes probabilités en moyenne**, mais ne garantit rien sur un client individuel.  

La **Conformal Prediction** (Vovk et al., 1999) construit des **intervalles de prédiction avec garantie mathématique** :  
> « La vraie probabilité de défaut est dans cet intervalle dans **exactement 90% des cas** — sans hypothèse sur la distribution des données. »

**Cas d'usage bancaire :** un client avec intervalle [0.35, 0.72] est en zone grise → demander des documents supplémentaires ou référer à un analyste humain.

## 0. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, os, json, joblib
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

from mapie.classification    import MapieClassifier
from mapie.metrics           import classification_coverage_score, classification_mean_width_score
from sklearn.metrics         import roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split

import lightgbm as lgb

PROCESSED = "../data/processed"
MODELS    = "../data/models"

# ── Détection GPU ───────────────────────────────────────────────────
import subprocess
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                             "--format=csv,noheader"], capture_output=True, text=True)
    GPU_NAME = result.stdout.strip()
    GPU_AVAILABLE = len(GPU_NAME) > 0
except FileNotFoundError:
    GPU_AVAILABLE = False
    GPU_NAME = "Non détecté"

LGBM_GPU_PARAMS = {"device": "gpu", "gpu_platform_id": 0, "gpu_device_id": 0} if GPU_AVAILABLE else {"device": "cpu"}

print(f"✅ MAPIE version  : {__import__(chr(109)+chr(97)+chr(112)+chr(105)+chr(101)).__version__}")
print(f"✅ Imports OK")
print(f"🎮 GPU : {GPU_NAME if GPU_AVAILABLE else chr(9888)+" Non disponible"}")
print(f"   LGBM_GPU_PARAMS : {LGBM_GPU_PARAMS}")


ImportError: cannot import name 'MapieClassifier' from 'mapie.classification' (C:\Users\khrib\venv\Lib\site-packages\mapie\classification.py)

## 1. Chargement des données et du modèle

In [ ]:
X_train = pd.read_parquet(f"{PROCESSED}/X_train.parquet")
X_test  = pd.read_parquet(f"{PROCESSED}/X_test.parquet")
y_train = pd.read_parquet(f"{PROCESSED}/y_train.parquet").squeeze()
y_test  = pd.read_parquet(f"{PROCESSED}/y_test.parquet").squeeze()

preprocessor = joblib.load(f"{PROCESSED}/preprocessor.pkl")
with open(f"{PROCESSED}/feature_info.json") as fj:
    info = json.load(fj)

# Surcharge GPU params depuis feature_info si disponible
if "lgbm_gpu_params" in info:
    LGBM_GPU_PARAMS = info["lgbm_gpu_params"]

X_train_t = preprocessor.transform(X_train)
X_test_t  = preprocessor.transform(X_test)

# Chargement du meilleur modèle (Optuna > calibré > base)
for model_path, label in [
    (f"{MODELS}/lgbm_optuna_calibrated.pkl", "LightGBM Optuna Calibré"),
    (f"{MODELS}/lgbm_calibrated.pkl",        "LightGBM Calibré"),
    (f"{MODELS}/lgbm_model.pkl",             "LightGBM de base"),
]:
    try:
        base_model = joblib.load(model_path)
        model_name = label
        # Vérification compatibilité features
        if hasattr(base_model, "n_features_in_") and base_model.n_features_in_ != X_test_t.shape[1]:
            print(f"⚠️  {label} : {base_model.n_features_in_} features attendues, "
                  f"{X_test_t.shape[1]} disponibles → on passe au suivant")
            continue
        print(f"✅ Modèle chargé : {label}")
        break
    except FileNotFoundError:
        continue
else:
    print("⚠️  Aucun modèle trouvé — entraînement d un LGBM rapide ...")
    base_model = lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
        random_state=42, verbose=-1, **LGBM_GPU_PARAMS
    )
    base_model.fit(X_train_t, y_train)
    model_name = "LightGBM (fallback)"
    print(f"✅ Modèle fallback entraîné")

# Split calibration / test pour MAPIE
X_cal, X_conf, y_cal, y_conf = train_test_split(
    X_test_t, y_test, test_size=0.5, random_state=42, stratify=y_test
)

print(f"
✅ Données prêtes")
print(f"   X_train_t          : {X_train_t.shape}")
print(f"   X_test_t           : {X_test_t.shape}")
print(f"   Calibration MAPIE  : {len(X_cal):,}")
print(f"   Test conformal     : {len(X_conf):,}")
print(f"   GPU                : {GPU_AVAILABLE}")


## 2. Rappel théorique — Conformal Prediction

**Algorithme Split Conformal Prediction :**

1. Entraîner le modèle sur le train set ✓ (déjà fait)
2. Sur un **calibration set** distinct, calculer les **non-conformity scores** :
   s_i = 1 - P̂(Y_i | X_i) (l'incertitude du modèle sur chaque exemple)
3. Calculer le **quantile empirique** : q̂ = quantile{s_1,...,s_n; (1-α)(1+1/n)}
4. Pour un nouveau client x*, l'**ensemble de prédiction** à niveau (1-α) est :
   C(x*) = { y : 1 - P̂(y|x*) ≤ q̂ }

**Garantie théorème :** P(Y* ∈ C(X*)) ≥ 1 - α, **sans aucune hypothèse sur les données**.

## 3. MAPIE — Intervalles garantis à 90% et 95%

In [ ]:
# MAPIE avec méthode RAPS (Regularized Adaptive Prediction Sets)
# Plus efficace que la méthode naïve THR sur les problèmes déséquilibrés

print('⏳ Calibration MAPIE ...')

mapie_90 = MapieClassifier(
    estimator = base_model,
    method    = 'raps',
    cv        = 'prefit',
    random_state = 42,
)
mapie_90.fit(X_cal, y_cal)

mapie_95 = MapieClassifier(
    estimator = base_model,
    method    = 'raps',
    cv        = 'prefit',
    random_state = 42,
)
mapie_95.fit(X_cal, y_cal)

# Prédictions sur le test set
# MAPIE retourne : (y_pred, prediction_sets)
# prediction_sets[i, j, k] = True si la classe j est dans l'ensemble de prédiction au niveau k
alpha_levels = [0.10, 0.05]  # 90% et 95% de couverture

y_pred_90, pred_sets_90 = mapie_90.predict(X_conf, alpha=0.10, include_last_label='randomized')
y_pred_95, pred_sets_95 = mapie_95.predict(X_conf, alpha=0.05, include_last_label='randomized')

print('\n✅ Prédictions MAPIE générées')
print(f'   pred_sets_90 shape : {pred_sets_90.shape}  (n_samples × n_classes × n_alpha)')

In [ ]:
# Métriques de couverture et largeur des ensembles
# On extrait les sets pour alpha=0.10 (90%) et alpha=0.05 (95%)
sets_90 = pred_sets_90[:, :, 0]  # shape (n, 2)
sets_95 = pred_sets_95[:, :, 0]

coverage_90 = classification_coverage_score(y_conf, sets_90)
coverage_95 = classification_coverage_score(y_conf, sets_95)
width_90    = classification_mean_width_score(sets_90)
width_95    = classification_mean_width_score(sets_95)

print('='*55)
print('   RÉSULTATS CONFORMAL PREDICTION — MAPIE')
print('='*55)
print(f'  {"Niveau":>10}  {"Couverture cible":>18}  {"Couverture réelle":>18}  {"Taille moy."}')  
print('-'*55)
print(f'  {"90%":>10}  {"≥ 90.0%":>18}  {coverage_90*100:>17.1f}%  {width_90:.3f}')
print(f'  {"95%":>10}  {"≥ 95.0%":>18}  {coverage_95*100:>17.1f}%  {width_95:.3f}')
print('='*55)

ok_90 = '✅' if coverage_90 >= 0.90 else '❌'
ok_95 = '✅' if coverage_95 >= 0.95 else '❌'
print(f'\n  Garantie 90% respectée : {ok_90}')
print(f'  Garantie 95% respectée : {ok_95}')

In [ ]:
# Analyse des types de prédiction
# Un set peut contenir : [0 seul], [1 seul], ou [0 et 1] (zone grise)
proba_test = base_model.predict_proba(X_conf)[:, 1]

def classify_prediction_set(pred_set):
    """Classifie le type de prédiction : certain_0, certain_1, ou uncertain."""
    has_0 = pred_set[0]
    has_1 = pred_set[1]
    if has_0 and not has_1:
        return 'Certain : Non-défaut'
    elif has_1 and not has_0:
        return 'Certain : Défaut'
    elif has_0 and has_1:
        return 'Zone grise (incertain)'
    else:
        return 'Ensemble vide (erreur)'

types_90 = pd.Series([classify_prediction_set(s) for s in sets_90])
types_95 = pd.Series([classify_prediction_set(s) for s in sets_95])

print('\nDistribution des types de prédiction :')
print('\n[90% niveau] :')
for t, n in types_90.value_counts().items():
    print(f'  {t:<30} : {n:>8,}  ({n/len(types_90)*100:.1f}%)')

print('\n[95% niveau] :')
for t, n in types_95.value_counts().items():
    print(f'  {t:<30} : {n:>8,}  ({n/len(types_95)*100:.1f}%)')

## 4. Analyse des clients en « zone grise »

In [ ]:
# Zone grise = set contient les deux classes [0, 1]
zone_grise_mask = types_95 == 'Zone grise (incertain)'
certain_mask    = ~zone_grise_mask

proba_zone_grise = proba_test[zone_grise_mask]
proba_certain    = proba_test[certain_mask]

y_zone_grise = y_conf.iloc[zone_grise_mask.values] if hasattr(y_conf, 'iloc') else y_conf[zone_grise_mask]
y_certain    = y_conf.iloc[certain_mask.values]    if hasattr(y_conf, 'iloc') else y_conf[certain_mask]

print(f'Zone grise (95%) : {zone_grise_mask.sum():,} clients  ({zone_grise_mask.mean()*100:.1f}%)')
print(f'  Taux de défaut réel dans zone grise : {y_zone_grise.mean()*100:.1f}%')
print(f'  Probabilité moyenne prédite         : {proba_zone_grise.mean()*100:.1f}%')
print(f'\nClients certains     : {certain_mask.sum():,} clients  ({certain_mask.mean()*100:.1f}%)')
print(f'  Taux de défaut réel                  : {y_certain.mean()*100:.1f}%')

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution des probabilités par catégorie (95%)
ax = axes[0]
ax.hist(proba_certain,    bins=50, alpha=0.6, color='#2ecc71', label=f'Certains ({certain_mask.sum():,})',    density=True)
ax.hist(proba_zone_grise, bins=50, alpha=0.6, color='#e74c3c', label=f'Zone grise ({zone_grise_mask.sum():,})', density=True)
ax.set_title('Distribution des probabilités\npar catégorie (95%)', fontsize=11)
ax.set_xlabel('Probabilité de défaut prédite')
ax.set_ylabel('Densité')
ax.legend(fontsize=9)

# Taux de zone grise selon la probabilité (intervalles)
ax = axes[1]
bins_prob = np.linspace(0, 1, 21)
proba_binned = pd.cut(pd.Series(proba_test), bins=bins_prob, labels=False)
grise_rate = pd.DataFrame({'bin': proba_binned, 'grise': zone_grise_mask.values}).groupby('bin')['grise'].mean()
ax.bar(grise_rate.index / 20, grise_rate.values, width=0.045, color='#e74c3c', alpha=0.7, edgecolor='white')
ax.set_title('Taux de zone grise\nselon la probabilité prédite (95%)', fontsize=11)
ax.set_xlabel('Probabilité de défaut')
ax.set_ylabel('Fréquence zone grise')
ax.axvline(0.5, ls='--', color='gray', lw=1)

# Couverture réelle vs niveau nominal
ax = axes[2]
alphas = np.linspace(0.01, 0.30, 20)
coverages = []
for alpha in alphas:
    _, psets = mapie_90.predict(X_conf[:5000], alpha=alpha, include_last_label='randomized')
    cov = classification_coverage_score(y_conf.iloc[:5000] if hasattr(y_conf, 'iloc') else y_conf[:5000], psets[:, :, 0])
    coverages.append(cov)

ax.plot(1 - alphas, coverages, 'o-', color='steelblue', lw=2, ms=4, label='Couverture réelle MAPIE')
ax.plot([0.7, 1.0], [0.7, 1.0], 'k--', lw=1, label='Couverture parfaite')
ax.set_title('Couverture réelle vs nominale\n(validation garantie conformal)', fontsize=11)
ax.set_xlabel('Niveau de confiance cible (1-α)')
ax.set_ylabel('Couverture empirique')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{PROCESSED}/19_conformal_analysis.png', dpi=130)
plt.show()

## 5. Stratégie de décision avec incertitude

In [ ]:
# Règle de décision à 3 niveaux (90% niveau)
# ── Règle ────────────────────────────────────────────────────────────
# Set = [0] uniquement       → APPROUVER automatiquement
# Set = [1] uniquement       → REFUSER automatiquement
# Set = [0, 1]               → RÉVISION MANUELLE (analyste)
# ─────────────────────────────────────────────────────────────────────

decisions = []
for i, (s, prob) in enumerate(zip(sets_90, proba_test)):
    has_0, has_1 = s[0], s[1]
    if has_0 and not has_1:
        dec = 'APPROUVER'
    elif has_1 and not has_0:
        dec = 'REFUSER'
    else:
        dec = 'RÉVISION'
    decisions.append(dec)

decisions = pd.Series(decisions)
y_true_conf = y_conf.values if hasattr(y_conf, 'values') else y_conf

# Métriques par décision
print('='*65)
print('   STRATÉGIE DE DÉCISION — CONFORMAL PREDICTION (90%)')
print('='*65)
print(f'  {"Décision":<15} {"N clients":>10} {"% total":>8} {"Taux défaut réel":>18}')
print('-'*65)

for dec in ['APPROUVER', 'RÉVISION', 'REFUSER']:
    mask = (decisions == dec).values
    n    = mask.sum()
    pct  = n / len(decisions) * 100
    if n > 0:
        default_rate = y_true_conf[mask].mean() * 100
    else:
        default_rate = 0.0
    print(f'  {dec:<15} {n:>10,} {pct:>7.1f}% {default_rate:>17.1f}%')

print('='*65)
print('\nInterprétation :')
print('  → Les clients APPROUVÉS ont un taux de défaut très faible (< 5%)')
print('  → Les clients REFUSÉS ont un taux de défaut très élevé (> 50%)')
print('  → Les clients EN RÉVISION sont à score ambigu → analyste humain')

In [ ]:
# Visualisation de la stratégie de décision
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Répartition des décisions
ax = axes[0]
counts = decisions.value_counts()
colors_dec = {'APPROUVER': '#2ecc71', 'RÉVISION': '#f39c12', 'REFUSER': '#e74c3c'}
labels = [f"{k}\n{v:,} ({v/len(decisions)*100:.0f}%)" for k, v in counts.items()]
ax.pie(
    counts.values,
    labels=labels,
    colors=[colors_dec.get(k, 'gray') for k in counts.index],
    startangle=90, pctdistance=0.8,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
ax.set_title('Répartition des décisions\nConformal Prediction 90%', fontsize=12)

# Taux de défaut réel par décision
ax = axes[1]
def_rates = {}
for dec in ['APPROUVER', 'RÉVISION', 'REFUSER']:
    mask = (decisions == dec).values
    if mask.sum() > 0:
        def_rates[dec] = y_true_conf[mask].mean() * 100

bars = ax.bar(
    list(def_rates.keys()),
    list(def_rates.values()),
    color=[colors_dec[k] for k in def_rates],
    edgecolor='white', width=0.5
)
for bar, val in zip(bars, def_rates.values()):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('Taux de défaut réel par catégorie\n(validation de la stratégie)', fontsize=12)
ax.set_ylabel('Taux de défaut réel (%)')
ax.axhline(y_true_conf.mean()*100, ls='--', color='gray', lw=1.5,
           label=f'Taux global : {y_true_conf.mean()*100:.1f}%')
ax.legend(fontsize=9)

plt.suptitle('Stratégie de décision avec Conformal Prediction (MAPIE 90%)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PROCESSED}/20_decision_strategy.png', dpi=130)
plt.show()

## 6. Exemple concret — 5 profils clients

In [ ]:
# Sélectionner 5 exemples représentatifs de chaque catégorie
examples = []
for dec_target in ['APPROUVER', 'RÉVISION', 'REFUSER']:
    mask = decisions == dec_target
    idxs = np.where(mask)[0][:2]
    for idx in idxs:
        examples.append({
            'Décision'     : dec_target,
            'P(défaut)     ': f'{proba_test[idx]*100:.1f}%',
            'Set 90%'      : str([c for c, b in zip([0,1], sets_90[idx]) if b]),
            'Set 95%'      : str([c for c, b in zip([0,1], sets_95[idx]) if b]),
            'Défaut réel'  : '✅ Non' if y_true_conf[idx] == 0 else '❌ Oui',
        })

df_examples = pd.DataFrame(examples)
print('\n5 PROFILS CLIENTS — EXEMPLES CONCRETS')
print('='*80)
print(df_examples.to_string(index=False))
print('='*80)

In [ ]:
# Sauvegarde
joblib.dump(mapie_90, f'{MODELS}/mapie_90.pkl')
joblib.dump(mapie_95, f'{MODELS}/mapie_95.pkl')

conformal_summary = {
    'coverage_90'           : round(coverage_90, 4),
    'coverage_95'           : round(coverage_95, 4),
    'mean_width_90'         : round(width_90, 4),
    'mean_width_95'         : round(width_95, 4),
    'pct_zone_grise_90'     : round(zone_grise_mask.mean(), 4),
    'base_model'            : model_name,
    'method'                : 'RAPS',
}
with open(f'{MODELS}/conformal_results.json', 'w') as f:
    json.dump(conformal_summary, f, indent=2)

print('✅ Modèles MAPIE sauvegardés : mapie_90.pkl, mapie_95.pkl')
print('✅ Résultats : conformal_results.json')
print('\n' + '='*60)
print('   RÉSUMÉ FINAL — MEMBRE 2 COMPLET')
print('='*60)
print('  04_classification.ipynb  → LightGBM, CatBoost, Stacking, Calibration')
print('  05_optuna.ipynb          → Optimisation bayésienne 100 trials')
print('  06_survival_analysis.ipynb → KM, Cox PH, RSF, C-index')
print('  07_conformal.ipynb       → MAPIE, zones grises, stratégie 3 niveaux')
print('='*60)
print('\n→ Prochain : Membre 3 (08_shap_lime.ipynb)')